<a href="https://colab.research.google.com/github/edwardcalvinvazc-maker/Music-Recommendation-Engine/blob/main/Music_Recommendation_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install hyperopt
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.model_selection import cross_val_score
import xgboost as xgb



In [4]:
music_data = pd.read_csv('/content/M+U_dataset.csv')
music_data.head()

,timestamp,user_id,age,gender,location,device_type,listening_time_mins,sessions_per_day,time_of_day,day_of_week,...,skip_count,added_to_playlist,finished_song,time_spent_on_song,repeat_count,first_time_listening,context_type,song_position_in_session,session_duration_mins,liked
0,2021-01-01 00:00:00,1040,35,Female,US,Mobile,3,2,Afternoon,Weekday,...,2,0,1,109,1,0,Relax,10,31,0
1,2021-01-01 00:30:00,1000,45,Male,US,Mobile,28,1,Evening,Weekday,...,1,1,1,116,1,0,Workout,6,38,0
2,2021-01-01 01:00:00,1025,35,Other,US,Mobile,65,3,Afternoon,Weekday,...,0,0,0,172,1,0,Workout,8,25,1
3,2021-01-01 01:30:00,1049,45,Male,US,Smart Speaker,2,1,Afternoon,Weekend,...,0,0,1,172,0,1,Relax,8,49,0
4,2021-01-01 02:00:00,1011,45,Male,UK,Mobile,44,1,Evening,Weekday,...,0,1,0,178,1,0,Relax,2,25,0


In [5]:
music_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70129 entries, 0 to 70128
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   timestamp                 70129 non-null  object 
 1   user_id                   70129 non-null  int64  
 2   age                       70129 non-null  int64  
 3   gender                    70129 non-null  object 
 4   location                  70129 non-null  object 
 5   device_type               70129 non-null  object 
 6   listening_time_mins       70129 non-null  int64  
 7   sessions_per_day          70129 non-null  int64  
 8   time_of_day               70129 non-null  object 
 9   day_of_week               70129 non-null  object 
 10  preferred_genre           70129 non-null  object 
 11  preferred_artist          70129 non-null  object 
 12  recent_skip_rate          70129 non-null  float64
 13  subscription_type         70129 non-null  object 
 14  song_i

In [6]:
print('Columns with string values (object dtype):')
print(pd.DataFrame(music_data.select_dtypes(include='object').columns.tolist()))

Columns with string values (object dtype):
                    0
0           timestamp
1              gender
2            location
3         device_type
4         time_of_day
5         day_of_week
6     preferred_genre
7    preferred_artist
8   subscription_type
9               title
10             artist
11              album
12              genre
13           language
14           explicit
15               mode
16   lyrics_sentiment
17        emotion_tag
18       context_type


In [7]:
print('Missing values in each column:')
display(music_data.isnull().sum())

Missing values in each column:


,0
timestamp,0
user_id,0
age,0
gender,0
location,0
device_type,0
listening_time_mins,0
sessions_per_day,0
time_of_day,0
day_of_week,0


In [8]:

# Make a copy to avoid modifying the original DataFrame directly if not desired
music_data_label_encoded = music_data.copy()

# Identify categorical columns (object type)
string_cols = music_data_label_encoded.select_dtypes(include='object').columns

# Apply Label Encoding to each string column
for col in string_cols:
    le = LabelEncoder()
    music_data_label_encoded[col] = le.fit_transform(music_data_label_encoded[col])

print('DataFrame after Label Encoding:')
display(music_data_label_encoded.head())

print('\nInfo of the Label Encoded DataFrame:')
music_data_label_encoded.info()

DataFrame after Label Encoding:


,timestamp,user_id,age,gender,location,device_type,listening_time_mins,sessions_per_day,time_of_day,day_of_week,...,skip_count,added_to_playlist,finished_song,time_spent_on_song,repeat_count,first_time_listening,context_type,song_position_in_session,session_duration_mins,liked
0,0,1040,35,0,4,1,3,2,0,0,...,2,0,1,109,1,0,2,10,31,0
1,1,1000,45,1,4,1,28,1,1,0,...,1,1,1,116,1,0,4,6,38,0
2,2,1025,35,2,4,1,65,3,0,0,...,0,0,0,172,1,0,4,8,25,1
3,3,1049,45,1,4,2,2,1,0,1,...,0,0,1,172,0,1,2,8,49,0
4,4,1011,45,1,3,1,44,1,1,0,...,0,1,0,178,1,0,2,2,25,0



Info of the Label Encoded DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70129 entries, 0 to 70128
Data columns (total 49 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   timestamp                 70129 non-null  int64  
 1   user_id                   70129 non-null  int64  
 2   age                       70129 non-null  int64  
 3   gender                    70129 non-null  int64  
 4   location                  70129 non-null  int64  
 5   device_type               70129 non-null  int64  
 6   listening_time_mins       70129 non-null  int64  
 7   sessions_per_day          70129 non-null  int64  
 8   time_of_day               70129 non-null  int64  
 9   day_of_week               70129 non-null  int64  
 10  preferred_genre           70129 non-null  int64  
 11  preferred_artist          70129 non-null  int64  
 12  recent_skip_rate          70129 non-null  float64
 13  subscription_type      

In [9]:
# 1. DEFINE PIPELINE BOUNDARIES
classifier_target = 'liked'
regressor_target = 'time_spent_on_song' # Assuming 'time_spent_on_song' is the intended regressor target

# Identify columns that are unique identifiers or should be excluded from features
# based on the blueprint's guidance.
text_id_columns = ['timestamp', 'user_id', 'song_id', 'title', 'artist', 'album', 'preferred_artist']

# 2. ELIMINATE TARGET LEAKAGE AND IDS
# Create a copy of the original music_data to work with
df_processed = music_data.copy()

# Separate target variables
y_class = df_processed[classifier_target]
y_reg = df_processed[regressor_target]

# Define all columns to be dropped from the feature matrix X
columns_to_drop = [classifier_target, regressor_target] + text_id_columns
X = df_processed.drop(columns=[col for col in columns_to_drop if col in df_processed.columns], errors='ignore')

print(f"Original DataFrame shape: {music_data.shape}")
print(f"Feature matrix (X) shape after dropping targets and IDs: {X.shape}")
print(f"Classification target (y_class) shape: {y_class.shape}")
print(f"Regression target (y_reg) shape: {y_reg.shape}")

Original DataFrame shape: (70129, 49)
Feature matrix (X) shape after dropping targets and IDs: (70129, 40)
Classification target (y_class) shape: (70129,)
Regression target (y_reg) shape: (70129,)


In [10]:
# 3. EXECUTE STRUCTURAL ENCODING
# Convert categorical features to Pandas 'category' types
# Using 'location' for 'user_region' as suggested by the blueprint
categorical_features = [
    'gender', 'location', 'device_type', 'time_of_day', 'day_of_week',
    'preferred_genre', 'subscription_type', 'genre', 'language', 'explicit',
    'mode', 'lyrics_sentiment', 'emotion_tag', 'context_type'
]

for col in categorical_features:
    if col in X.columns:
        X[col] = X[col].astype('category')

print('\nFeatures DataFrame (X) after converting categorical columns to `category` dtype:')
X.info()


Features DataFrame (X) after converting categorical columns to `category` dtype:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70129 entries, 0 to 70128
Data columns (total 40 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   age                       70129 non-null  int64   
 1   gender                    70129 non-null  category
 2   location                  70129 non-null  category
 3   device_type               70129 non-null  category
 4   listening_time_mins       70129 non-null  int64   
 5   sessions_per_day          70129 non-null  int64   
 6   time_of_day               70129 non-null  category
 7   day_of_week               70129 non-null  category
 8   preferred_genre           70129 non-null  category
 9   recent_skip_rate          70129 non-null  float64 
 10  subscription_type         70129 non-null  category
 11  genre                     70129 non-null  category
 12  release_year        

In [11]:
# Perform the train-test split on features (X) and both target variables (y_class, y_reg)
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg,
    test_size=0.20,       # 20% reserved for unbiased testing
    random_state=42       # Reproducible shuffling seed
)

print(f"\nPipeline Preparation Complete.")
print(f"Training Matrix Dimensions: {X_train.shape}")
print(f"Testing Matrix Dimensions: {X_test.shape}")
print(f"Classification Training Target Dimensions: {y_class_train.shape}")
print(f"Classification Testing Target Dimensions: {y_class_test.shape}")
print(f"Regression Training Target Dimensions: {y_reg_train.shape}")
print(f"Regression Testing Target Dimensions: {y_reg_test.shape}")


Pipeline Preparation Complete.
Training Matrix Dimensions: (56103, 40)
Testing Matrix Dimensions: (14026, 40)
Classification Training Target Dimensions: (56103,)
Classification Testing Target Dimensions: (14026,)
Regression Training Target Dimensions: (56103,)
Regression Testing Target Dimensions: (14026,)


## Hyperparameter Tuning for XGBoost Classifier with Hyperopt

In [21]:
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import numpy as np

# Define the search space for XGBoost Classifier
space_clf = {
    'n_estimators': hp.quniform('n_estimators', 100, 500, 50),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'gamma': hp.uniform('gamma', 0.0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha', 0.0, 0.5),
    'reg_lambda': hp.uniform('reg_lambda', 0.0, 0.5)
}

In [23]:
# Define the objective function for the classifier
def objective_clf(params):
    # Create a copy of params to avoid modifying the original dictionary
    # and remove keys that will be explicitly passed as integers.
    clf_params = params.copy()
    n_estimators = int(clf_params.pop('n_estimators'))
    max_depth = int(clf_params.pop('max_depth'))

    clf = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        **clf_params,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1
    )

    # Use cross-validation to evaluate the model
    score = cross_val_score(clf, X_train, y_class_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()

    # Hyperopt minimizes the loss, so we return 1 - score for ROC AUC
    return {'loss': 1 - score, 'status': STATUS_OK}

# Run the Hyperopt search for the classifier
trials_clf = Trials()
best_clf = fmin(
    fn=objective_clf,
    space=space_clf,
    algo=tpe.suggest,
    max_evals=50, # Number of evaluations, similar to n_iter in RandomizedSearchCV
    trials=trials_clf,
    rstate=np.random.default_rng(42) # For reproducibility
)

print("Hyperopt optimization for XGBoost Classifier complete.")

100%|██████████| 50/50 [12:07<00:00, 14.56s/trial, best loss: 0.4991022911946129]
Hyperopt optimization for XGBoost Classifier complete.


In [24]:
print("Best hyperparameters for XGBoost Classifier:", best_clf)

# To get the best ROC AUC score, we need to re-evaluate with the best parameters
best_params_clf = {
    'n_estimators': int(best_clf['n_estimators']),
    'max_depth': int(best_clf['max_depth']),
    'learning_rate': best_clf['learning_rate'],
    'subsample': best_clf['subsample'],
    'colsample_bytree': best_clf['colsample_bytree'],
    'gamma': best_clf['gamma'],
    'reg_alpha': best_clf['reg_alpha'],
    'reg_lambda': best_clf['reg_lambda'],
}

best_clf_model = xgb.XGBClassifier(
    **best_params_clf,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

best_score_clf = cross_val_score(best_clf_model, X_train, y_class_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()
print(f"Best ROC AUC score with Hyperopt: {best_score_clf:.4f}")

Best hyperparameters for XGBoost Classifier: {'colsample_bytree': np.float64(0.6730438114891641), 'gamma': np.float64(0.18458877217294783), 'learning_rate': np.float64(0.021926401031553647), 'max_depth': np.float64(6.0), 'n_estimators': np.float64(150.0), 'reg_alpha': np.float64(0.2763791522948141), 'reg_lambda': np.float64(0.17970223558255694), 'subsample': np.float64(0.6618131913710332)}
Best ROC AUC score with Hyperopt: 0.5009


## Hyperparameter Tuning for XGBoost Regressor with Hyperopt

In [25]:
# Define the search space for XGBoost Regressor
space_reg = {
    'n_estimators': hp.quniform('n_estimators_reg', 100, 500, 50),
    'learning_rate': hp.loguniform('learning_rate_reg', np.log(0.01), np.log(0.2)),
    'max_depth': hp.quniform('max_depth_reg', 3, 10, 1),
    'subsample': hp.uniform('subsample_reg', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree_reg', 0.6, 1.0),
    'gamma': hp.uniform('gamma_reg', 0.0, 0.5),
    'reg_alpha': hp.uniform('reg_alpha_reg', 0.0, 0.5),
    'reg_lambda': hp.uniform('reg_lambda_reg', 0.0, 0.5)
}

In [28]:
# Define the objective function for the regressor
def objective_reg(params):
    params_int = {
        'n_estimators': int(params['n_estimators']),
        'max_depth': int(params['max_depth'])
    }

    reg = xgb.XGBRegressor(
        **params,
        **params_int,
        objective='reg:squarederror',
        tree_method='hist',
        enable_categorical=True,
        random_state=42,
        n_jobs=-1
    )

    # Use cross-validation to evaluate the model with negative RMSE
    # Hyperopt minimizes loss, so we want to minimize -neg_root_mean_squared_error (which is RMSE)
    score = cross_val_score(reg, X_train, y_reg_train, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()

    # cross_val_score returns negative RMSE for 'neg_root_mean_squared_error', so we need to negate it to get RMSE
    rmse = -score

    return {'loss': rmse, 'status': STATUS_OK}

# Run the Hyperopt search for the regressor
trials_reg = Trials()
best_reg = fmin(
    fn=objective_reg,
    space=space_reg,
    algo=tpe.suggest,
    max_evals=50, # Number of evaluations
    trials=trials_reg,
    rstate=np.random.default_rng(42) # For reproducibility
)

print("Hyperopt optimization for XGBoost Regressor complete.")

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

ERROR:hyperopt.fmin:job exception: xgboost.sklearn.XGBRegressor() got multiple values for keyword argument 'n_estimators'


  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]


TypeError: xgboost.sklearn.XGBRegressor() got multiple values for keyword argument 'n_estimators'

In [ ]:
print("Best hyperparameters for XGBoost Regressor:", best_reg)

# To get the best RMSE score, we need to re-evaluate with the best parameters
best_params_reg = {
    'n_estimators': int(best_reg['n_estimators_reg']),
    'max_depth': int(best_reg['max_depth_reg']),
    'learning_rate': best_reg['learning_rate_reg'],
    'subsample': best_reg['subsample_reg'],
    'colsample_bytree': best_reg['colsample_bytree_reg'],
    'gamma': best_reg['gamma_reg'],
    'reg_alpha': best_reg['reg_alpha_reg'],
    'reg_lambda': best_reg['reg_lambda_reg'],
}

best_reg_model = xgb.XGBRegressor(
    **best_params_reg,
    objective='reg:squarederror',
    tree_method='hist',
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

best_score_reg = cross_val_score(best_reg_model, X_train, y_reg_train, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()
print(f"Best RMSE score with Hyperopt: {-best_score_reg:.4f}")

In [ ]:
import xgboost as xgb

# Initialize the XGBoost Classifier with the best parameters from RandomizedSearchCV
xgb_classifier = xgb.XGBClassifier(
    **random_search_clf.best_params_,
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    enable_categorical=True,
    random_state=42
)

# Train the classifier
xgb_classifier.fit(X_train, y_class_train)

print("XGBoost Classifier training complete with best hyperparameters.")

In [ ]:

# 1. Initialize the baseline regressor with your native category setting
base_reg = xgb.XGBRegressor(
    tree_method="hist",
    enable_categorical=True,
    objective="reg:squarederror"
)

# 2. Define the hyperparameter grid distributions
param_distributions_reg = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# 3. Set up the automated cross-validation searcher
tuned_reg_search = RandomizedSearchCV(
    estimator=base_reg,
    param_distributions=param_distributions_reg,
    n_iter=10,
    scoring='neg_root_mean_squared_error',  # Optimizing to minimize your timing errors
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# 4. Fit the search space using your continuous training array targets
print("Starting Regressor Hyperparameter Optimization Search...")
tuned_reg_search.fit(X_train, y_reg_train)

print("\n=== REGRESSOR TUNING COMPLETE ===")
print("Best Negative RMSE Score Found:", tuned_reg_search.best_score_)
print("Best Regressor Hyperparameters :", tuned_reg_search.best_params_)